# Notebook 4: Analyze Xenium Data with Squidpy

**Dataset:** 10x Genomics Xenium - Human Lung Cancer (FFPE, Multimodal Cell Segmentation)

**Reference:** [Squidpy Xenium Tutorial](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_xenium.html)

This notebook demonstrates analysis of Xenium spatial transcriptomics data including QC, clustering, spatial statistics (centrality scores, co-occurrence, neighborhood enrichment, Moran's I).

## 1. Setup & Imports

In [ ]:
import spatialdata as sd
from spatialdata_io import xenium

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq

## 2. Load Xenium Data

Use `spatialdata-io` to parse the Xenium dataset into a SpatialData object.

In [ ]:
# Specify paths
xenium_path = "./Xenium"
zarr_path = "./Xenium.zarr"

# Read and convert to Zarr
sdata = xenium(xenium_path)
sdata.write(zarr_path)

# Read from zarr store
sdata = sd.read_zarr(zarr_path)
sdata

In [ ]:
# Extract the AnnData table
adata = sdata.tables["table"]
adata

In [ ]:
# Check obs metadata
adata.obs.head()

In [ ]:
# Spatial coordinates
adata.obsm["spatial"]

## 3. Quality Control Metrics

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
# Control probe percentages
cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])

axs[1].set_title("Unique transcripts per cell")
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, ax=axs[1])

axs[2].set_title("Area of segmented cells")
sns.histplot(adata.obs["cell_area"], kde=False, ax=axs[2])

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False, ax=axs[3],
)
plt.tight_layout()
plt.savefig("../results/04_xenium/01_qc_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Filtering and Preprocessing

In [ ]:
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata)

## 5. Visualize Annotation on UMAP and Spatial Coordinates

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "leiden"],
    wspace=0.4,
)
plt.savefig("../results/04_xenium/02_umap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=["leiden"],
    wspace=0.4,
)
plt.savefig("../results/04_xenium/03_spatial_leiden.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Spatial Statistics

### 6.1 Centrality Scores

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)
sq.gr.centrality_scores(adata, cluster_key="leiden")
sq.pl.centrality_scores(adata, cluster_key="leiden", figsize=(16, 5))
plt.savefig("../results/04_xenium/04_centrality.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.2 Co-occurrence Probability

In [ ]:
sdata.tables["subsample"] = sc.pp.subsample(adata, fraction=0.5, copy=True)
adata_subsample = sdata.tables["subsample"]

In [ ]:
sq.gr.co_occurrence(
    adata_subsample,
    cluster_key="leiden",
)
sq.pl.co_occurrence(
    adata_subsample,
    cluster_key="leiden",
    clusters="12",
    figsize=(10, 10),
)
plt.savefig("../results/04_xenium/07_co_occurrence.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sq.pl.spatial_scatter(
    adata_subsample,
    color="leiden",
    shape=None,
    size=2,
)
plt.show()

### 6.3 Neighborhood Enrichment

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="leiden")

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
sq.pl.nhood_enrichment(
    adata,
    cluster_key="leiden",
    figsize=(8, 8),
    title="Neighborhood enrichment adata",
    ax=ax[0],
)
sq.pl.spatial_scatter(adata_subsample, color="leiden", shape=None, size=2, ax=ax[1])
plt.savefig("../results/04_xenium/05_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.4 Moran's I Score

In [ ]:
sq.gr.spatial_neighbors(adata_subsample, coord_type="generic", delaunay=True)
sq.gr.spatial_autocorr(
    adata_subsample,
    mode="moran",
    n_perms=100,
    n_jobs=1,
)
adata_subsample.uns["moranI"].head(10)

In [ ]:
sq.pl.spatial_scatter(
    adata_subsample,
    library_id="spatial",
    color=["AREG", "MET"],
    shape=None,
    size=2,
    img=False,
)
plt.savefig("../results/04_xenium/06_moranI_genes.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

- Loaded Xenium human lung cancer data (161,000 cells × 480 genes)
- Performed QC: control probe % < 0.01%, confirming high data quality
- Filtered cells (min 10 counts) and genes (min 5 cells)
- Normalized, PCA, UMAP, Leiden clustering
- Computed spatial statistics:
  - **Centrality scores** (closeness, degree, clustering coefficient)
  - **Co-occurrence probability** across increasing radii
  - **Neighborhood enrichment** (z-score based permutation test)
  - **Moran's I** → top spatially variable genes: AREG, MET, ANXA1, EPCAM